In [4]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras import layers

# 1. Load your cleaned dataset
df = pd.read_csv('cleaned_DOAS_urdaneta_2014_2021.csv')

# Ensure date is sorted
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# 2. Select the 8 input features and target (AQI)
feature_cols = ['year', 'month', 'day', 'so2', 'nox', 'o3', 'pm10', 'co']
target_col = 'aqi'

# Scale features and target between 0 and 1
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(df[feature_cols])
y_scaled = scaler_y.fit_transform(df[[target_col]])

# 3. Define the sliding window parameter (e.g., past 14 days to predict next day)
window = 14  # <-- Fixes the undefined 'window' squiggle
n_features = len(feature_cols)  # equals 8


def create_sequences(X, y, window_size):
  X_seq, y_seq = [], []
  for i in range(len(X) - window_size):
    X_seq.append(X[i : i + window_size])
    y_seq.append(y[i + window_size])
  return np.array(X_seq), np.array(y_seq)


X_seq, y_seq = create_sequences(X_scaled, y_scaled, window)

# 4. Chronological Train-Test Split (80/20)
split = int(len(X_seq) * 0.8)
X_train, X_test = X_seq[:split], X_seq[split:]
y_train, y_test = y_seq[:split], y_seq[split:]

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.optimizers import Adam
# Model Definition
model = tf.keras.Sequential([
    # Input shape: (Time Steps, Features) -> (window, 8)
    layers.Input(shape=(window, n_features)),
    # First GRU layer
    layers.GRU(128, return_sequences=True),
    layers.Dropout(0.3),
    # Second GRU layer
    layers.GRU(64, return_sequences=True),
    layers.Dropout(0.2),
    # Third GRU layer
    layers.GRU(32),
    layers.Dropout(0.1),
    # Dense layer
    layers.Dense(16, activation='relu'),
    # Output Layer (predicting continuous AQI)
    layers.Dense(1, activation='linear'),
])

# Compile
model.compile(
    optimizer=Adam(learning_rate=0.001),
        loss='mae',
        metrics=['mae','mse']
)

# Callbacks for early stopping and best checkpoint saving
callbacks = [
    EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
    ),
    ModelCheckpoint(
        filepath='best_gru_aqi_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1,
    ),
]

# Train
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=128,verbose=1,,
    callbacks=callbacks,
)

Epoch 1/100
72/73 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: nan - mae: nan
Epoch 1: val_loss improved from None to nan, saving model to best_gru_aqi_model.keras

Epoch 1: finished saving model to best_gru_aqi_model.keras
73/73 ━━━━━━━━━━━━━━━━━━━━ 10s 38ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 2/100
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: nan - mae: nan
Epoch 2: val_loss did not improve from nan
73/73 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 3/100
72/73 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: nan - mae: nan
Epoch 3: val_loss did not improve from nan
73/73 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 4/100
72/73 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: nan - mae: nan
Epoch 4: val_loss did not improve from nan
73/73 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 5/100
72/73 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/s

KeyboardInterrupt: 

In [ ]:
# Predict on test set
y_pred_scaled = model.predict(X_test)

# Invert scale back to real AQI units
y_pred = scaler_y.inverse_transform(y_pred_scaled)
y_true = scaler_y.inverse_transform(y_test)

# Evaluate
mae = np.mean(np.abs(y_true - y_pred))
rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
print(f'Test AQI MAE:  {mae:.2f}')
print(f'Test AQI RMSE: {rmse:.2f}')